# Run This To Reproduce The Project Checks

These cells run the actual commands again so the results can be reproduced.

Run cells from top to bottom. The slowest parts are package installation and downloading the real taxi data.

## 1. Confirm We Are In The Project Folder

This should show files like `README.md`, `pyproject.toml`, `src`, `tests`, and `scripts`.

In [7]:
from pathlib import Path

print("Current folder:", Path.cwd())
required = ["README.md", "pyproject.toml", "src", "tests", "scripts", "docs"]
for item in required:
    print(f"{item:15} exists:", Path(item).exists())

Current folder: C:\Users\wongm\ORIE5270\Final project - Copy
README.md       exists: True
pyproject.toml  exists: True
src             exists: True
tests           exists: True
scripts         exists: True
docs            exists: True


## 2. Install The Project

If it shows pink/red dependency warnings about unrelated Anaconda packages, that is usually okay as long as the final line says the project installed successfully.

This can take a few minutes.

In [11]:
import sys

!{sys.executable} -m pip install -e ".[dev]" coverage

Defaulting to user installation because normal site-packages is not writeable

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
daal4py 2021.5.0 requires daal==2021.4.0, which is not installed.
numba 0.55.1 requires numpy<1.22,>=1.18, but you have numpy 1.26.4 which is incompatible.
scipy 1.7.3 requires numpy<1.23.0,>=1.16.5, but you have numpy 1.26.4 which is incompatible.



Obtaining file:///C:/Users/wongm/ORIE5270/Final%20project%20-%20Copy
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 15.8/15.8 MB 43.2 MB/s  0:00:00
  Building editable for orie5270-final-project (pyproject.toml): started
  Building editable for orie5270-final-project (pyproject.toml): finished with status 'done'
  Created wheel for orie5270-final-project: filename=orie5270_final_project-0.1.0-0.editable-py3-none-any.whl size=5486 sha256=ddb35140e43b539f38e7b9ec96c8db74a45ee0071ff0b3c88c0eb4d6c49df585


## 3. Run Unit Tests With Coverage

Expected final result:

```text
34 passed
Required test coverage of 80% reached
Total coverage: about 99%
```

In [9]:
import os
import sys
import subprocess

env = os.environ.copy()
# This avoids slow/hanging pytest plugins from Anaconda/Jupyter.
env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"

commands = [
    [sys.executable, "-m", "coverage", "run", "--source=orie5270_project", "-m", "pytest", "tests", "-q", "-o", "addopts="],
    [sys.executable, "-m", "coverage", "report", "--fail-under=80"],
]

for command in commands:
    print("\nRunning:", " ".join(command))
    result = subprocess.run(
        command,
        env=env,
        text=True,
        capture_output=True,
        timeout=180,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print("Exit code:", result.returncode)
    if result.returncode != 0:
        raise RuntimeError("Command failed; see output above.")



Running: C:\ProgramData\Anaconda3\python.exe -m coverage run --source=orie5270_project -m pytest tests -q -o addopts=
..................................                                       [100%]
34 passed in 13.88s

Exit code: 0

Running: C:\ProgramData\Anaconda3\python.exe -m coverage report --fail-under=80
Name                              Stmts   Miss  Cover
-----------------------------------------------------
src\orie5270_project\cli.py          94      2    98%
src\orie5270_project\dataset.py      23      0   100%
src\orie5270_project\metrics.py      21      0   100%
src\orie5270_project\taxi.py         84      0   100%
-----------------------------------------------------
TOTAL                               222      2    99%

Exit code: 0


## 4. Run The Small Demo

This does not need the real taxi data.

In [12]:
!python scripts/run_demo.py

Missing-value summary:
               missing_count  missing_rate
timestamp                  0      0.000000

C:\Users\wongm\AppData\Roaming\Python\Python39\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\wongm\AppData\Roaming\Python\Python39\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (



demand                     0      0.000000
weather_score              1      0.166667

Train rows: 4
Test rows: 2
Baseline MAE: 3.75
Baseline RMSE: 3.783


## 5. Check Whether Real Data Is Already Present

If these are `False`, run the download cell in the next section. If they are `True`, skip download and run the analysis.

In [13]:
from pathlib import Path

data_files = [
    Path("data/yellow_tripdata_2025-01.parquet"),
    Path("data/yellow_tripdata_2025-02.parquet"),
    Path("data/taxi_zone_lookup.csv"),
]
for path in data_files:
    print(f"{path}:", path.exists())

data\yellow_tripdata_2025-01.parquet: True
data\yellow_tripdata_2025-02.parquet: True
data\taxi_zone_lookup.csv: True


## 6. Download Real Data If Needed

Only run this if the previous cell showed missing data. This downloads about 120 MB from the public NYC TLC source, so it may take time.

In [ ]:
# Run this only if the data files above are missing.
!python scripts/download_data.py

## 7. Run The Full Taxi Analysis

This reads the real dataset, creates borough-level hourly demand, evaluates forecasting baselines, and writes CSV/PNG outputs.

In [14]:
!python scripts/run_taxi_analysis.py

Model comparison metrics by borough:
  borough              model  train_rows  test_rows      mae     rmse
    Bronx seasonal_naive_24h        1248        168   13.815   18.583
    Bronx     hourly_profile        1248        168   15.374   20.771
 Brooklyn seasonal_naive_24h        1248        168   56.970   80.473
 Brooklyn     hourly_profile        1248        168   57.530   80.664
Manhattan     hourly_profile        1253        168 1221.434 1655.995
Manhattan seasonal_naive_24h        1253        168 1336.321 2138.652
   Queens     hourly_profile        1251        168   85.212  110.595
   Queens seasonal_naive_24h        1251        168   87.143  123.386

Saved metrics to:     C:\Users\wongm\ORIE5270\Final project - Copy\data\processed\borough_model_metrics.csv
Saved predictions to: C:\Users\wongm\ORIE5270\Final project - Copy\data\processed\borough_model_predictions.csv
Saved figures to:     C:\Users\wongm\ORIE5270\Final project - Copy\docs\figures


C:\Users\wongm\AppData\Roaming\Python\Python39\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\wongm\AppData\Roaming\Python\Python39\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


## 8. Read The Generated Metrics

This reads the actual CSV created by the analysis command above. It is not hardcoded.

In [ ]:
import pandas as pd

metrics = pd.read_csv("data/processed/borough_model_metrics.csv")
metrics

## 9. Display Generated Figures

These are the actual PNG files from `docs/figures/`.

In [ ]:
from IPython.display import Image, display

display(Image("docs/figures/borough_model_comparison.png"))
display(Image("docs/figures/manhattan_forecast_last_72_hours.png"))

## 10. Import Check

This verifies that the package imports work after installation.

In [ ]:
from orie5270_project.taxi import load_yellow_taxi_data, load_zone_lookup
from orie5270_project.metrics import mae, rmse

print("Imports work")

## Final Check

If the test cell passes, the demo runs, the full analysis runs, and the metrics/figures display, then the project is reproducible and satisfies the grading requirements.